In [43]:
from pathlib import Path
import re

In [44]:
text = Path("../data/fil_ag.md").read_text(encoding="utf-8")

In [45]:
def normalize_markup(line: str) -> str:
    line = line.strip()

    # **## ...** → ## ...
    if line.startswith("**") and line.endswith("**"):
        line = line[2:-2].strip()

    return line

def is_chapter_no(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^## Глава\s+\d+", line))


def is_section(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^## \d+\.\d+\s+", line))


def is_chapter_name(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^## ", line))


def is_definition(line: str) -> bool:
    return bool(
        re.match(r"^∙\s+.*?\bназывается\b", line)
    )

def is_theorem(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^\*\*Теорема\b", line))

def is_proof_start(line: str) -> bool:
    return line.strip().startswith("♦")


def is_proof_end(line: str) -> bool:
    return line.strip().endswith("⊠")

def is_start_object(line_type: str) -> bool:
    return line_type == "chapter_no" or line_type == "chapter_name" or line_type == "section" or line_type == "theorem" or line_type == "definition"

In [46]:
def classify_line(line: str) -> str:

    if is_chapter_no(line):
        return "chapter_no"

    elif is_section(line):
        return "section"

    elif is_chapter_name(line):
        return "chapter_name"

    elif is_theorem(line):
        return "theorem"

    elif is_definition(line):
        return "definition"

    return "text"

In [47]:
blocks = []

current_chapter = ""
current_chapter_name = ""
current_section = ""

current_object = ""
current_object_text = ""

for line in text.splitlines():

    if line.strip() == "":
        continue

    line_type = classify_line(line)

    if line_type in ("chapter_no", "chapter_name", "section"):
        parsed_line = re.sub(r"^\**##\s*", "", line)
    else:
        parsed_line = line

    if line_type == "text":
        current_object_text += parsed_line
        continue
    
    elif line_type == "chapter_no":
        current_chapter = parsed_line

    elif line_type == "chapter_name":
        current_chapter_name = parsed_line

    elif line_type == "section":
        current_section = parsed_line

    blocks.append({
        "CHAPTER": current_chapter,
        "CHAPTER_NAME": current_chapter_name,
        "SECTION": current_section,
        "TYPE": current_object,
        "TEXT": current_object_text
    })

    current_object = line_type
    current_object_text = parsed_line

In [48]:
blocks = []

current_chapter = ""
current_chapter_name = ""
current_section = ""

current_object_type = ""
current_object_text = ""

def is_new_block(line_type: str) -> bool:
    return line_type in ("chapter_no", "chapter_name", "section", "definition", "theorem")

for line in text.splitlines():
    if line.strip() == "":
        continue
    
    line_type = classify_line(line)

    if line_type in ("chapter_no", "chapter_name", "section"):
        parsed_line = re.sub(r"^\**##\s*", "", line)
    else:
        parsed_line = line

    if line_type == "chapter_no":
        current_chapter = parsed_line

    elif line_type == "chapter_name":
        current_chapter_name = parsed_line

    elif line_type == "section":
        current_section = parsed_line

    elif line_type in ("definition", "theorem"):
        if current_object_type != "":
            blocks.append({
                    "CHAPTER": current_chapter,
                    "CHAPTER_NAME": current_chapter_name,
                    "SECTION": current_section,
                    "TYPE": current_object_type,
                    "TEXT": current_object_text
                })

        current_object_text = parsed_line
        current_object_type = line_type

    else:
        current_object_text += parsed_line

In [49]:
for block in blocks[:30]:
    print(block)

{'CHAPTER': 'Глава 1', 'CHAPTER_NAME': 'Метод координат', 'SECTION': '1.1 Величина направленного отрезка. Теорема Шаля. Декартова система координат на прямой.', 'TYPE': 'definition', 'TEXT': '∙ Отрезком называется часть прямой, ограниченная двумя точками. Отрезок называется направленным, если указано, какая из граничных точек является начальной и какая конечной (обозначения: $\\overline{AB}$ — направленный отрезок; $|\\overline{AB}|$ — длина направленного отрезка).'}
{'CHAPTER': 'Глава 1', 'CHAPTER_NAME': 'Метод координат', 'SECTION': '1.1 Величина направленного отрезка. Теорема Шаля. Декартова система координат на прямой.', 'TYPE': 'definition', 'TEXT': '∙ Отрезок называется нулевым, если его начальная и конечная точки совпадают. Длина нулевого направленного отрезка равна нулю.'}
{'CHAPTER': 'Глава 1', 'CHAPTER_NAME': 'Метод координат', 'SECTION': '1.1 Величина направленного отрезка. Теорема Шаля. Декартова система координат на прямой.', 'TYPE': 'definition', 'TEXT': '∙ Прямая линия с